# StyleGAN Resolution Capping Demo

This notebook demonstrates how to load a pre-trained StyleGAN2 or StyleGAN3 model and create a new generator with a capped resolution (e.g., maximum 512px).

In [ ]:
import os
import torch
import numpy as np
import PIL.Image
import matplotlib.pyplot as plt

# Make sure we're in the StyleGAN3 directory
os.chdir('/workspace/stylegan3')

import dnnlib
import legacy
from cap_resolution_utils import (
    load_and_cap_generator,
    generate_images,
    save_generator
)

## Configuration

In [ ]:
# Choose a pre-trained model
# StyleGAN3 model
stylegan3_pkl = "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"

# StyleGAN2 model
stylegan2_pkl = "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan2/versions/1/files/stylegan2-ffhq-1024x1024.pkl"

# Set the target resolution
target_resolution = 512

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## StyleGAN3 Example

In [ ]:
# Load and cap a StyleGAN3 model
print(f"Loading and capping StyleGAN3 model...")
G3_original, G3_capped = load_and_cap_generator(
    stylegan3_pkl, 
    target_resolution, 
    is_stylegan3=True
)

print(f"Original StyleGAN3 resolution: {G3_original.img_resolution}")
print(f"Capped StyleGAN3 resolution: {G3_capped.img_resolution}")

In [ ]:
# Generate images with both generators
seed = 42
z = torch.from_numpy(np.random.RandomState(seed).randn(1, G3_original.z_dim)).to(device)

# Generate images
img3_original = generate_images(G3_original, z, truncation_psi=0.7)
img3_capped = generate_images(G3_capped, z, truncation_psi=0.7)

# Convert to displayable format
img3_original_display = (img3_original.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
img3_capped_display = (img3_capped.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)

# Display the images side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

ax1.imshow(img3_original_display[0].cpu().numpy())
ax1.set_title(f"StyleGAN3 Original ({G3_original.img_resolution}x{G3_original.img_resolution})")
ax1.axis('off')

ax2.imshow(img3_capped_display[0].cpu().numpy())
ax2.set_title(f"StyleGAN3 Capped ({G3_capped.img_resolution}x{G3_capped.img_resolution})")
ax2.axis('off')

plt.tight_layout()
plt.show()

## StyleGAN2 Example

In [ ]:
# Load and cap a StyleGAN2 model
print(f"Loading and capping StyleGAN2 model...")
G2_original, G2_capped = load_and_cap_generator(
    stylegan2_pkl, 
    target_resolution, 
    is_stylegan3=False
)

print(f"Original StyleGAN2 resolution: {G2_original.img_resolution}")
print(f"Capped StyleGAN2 resolution: {G2_capped.img_resolution}")

In [ ]:
# Generate images with both generators
seed = 42
z = torch.from_numpy(np.random.RandomState(seed).randn(1, G2_original.z_dim)).to(device)

# Generate images
img2_original = generate_images(G2_original, z, truncation_psi=0.7)
img2_capped = generate_images(G2_capped, z, truncation_psi=0.7)

# Convert to displayable format
img2_original_display = (img2_original.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
img2_capped_display = (img2_capped.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)

# Display the images side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

ax1.imshow(img2_original_display[0].cpu().numpy())
ax1.set_title(f"StyleGAN2 Original ({G2_original.img_resolution}x{G2_original.img_resolution})")
ax1.axis('off')

ax2.imshow(img2_capped_display[0].cpu().numpy())
ax2.set_title(f"StyleGAN2 Capped ({G2_capped.img_resolution}x{G2_capped.img_resolution})")
ax2.axis('off')

plt.tight_layout()
plt.show()

## Try Different Seeds

In [ ]:
def generate_and_display(generator_type, seed, truncation_psi=0.7):
    """Generate and display images from both original and capped generators."""
    if generator_type.lower() == 'stylegan3':
        G_original, G_capped = G3_original, G3_capped
        title = "StyleGAN3"
    else:
        G_original, G_capped = G2_original, G2_capped
        title = "StyleGAN2"
    
    # Create a random latent vector
    z = torch.from_numpy(np.random.RandomState(seed).randn(1, G_original.z_dim)).to(device)
    
    # Generate images
    img_original = generate_images(G_original, z, truncation_psi=truncation_psi)
    img_capped = generate_images(G_capped, z, truncation_psi=truncation_psi)
    
    # Convert to displayable format
    img_original_display = (img_original.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    img_capped_display = (img_capped.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    
    # Display the images side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    ax1.imshow(img_original_display[0].cpu().numpy())
    ax1.set_title(f"{title} Original ({G_original.img_resolution}x{G_original.img_resolution})")
    ax1.axis('off')
    
    ax2.imshow(img_capped_display[0].cpu().numpy())
    ax2.set_title(f"{title} Capped ({G_capped.img_resolution}x{G_capped.img_resolution})")
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Try with different seeds for StyleGAN3
generate_and_display('stylegan3', seed=100)

In [ ]:
# Try with different seeds for StyleGAN2
generate_and_display('stylegan2', seed=100)

In [ ]:
# Try with another seed and different truncation
generate_and_display('stylegan3', seed=200, truncation_psi=0.5)

## Save the Capped Generators

In [ ]:
# Save the capped StyleGAN3 generator
stylegan3_output_pkl = f"stylegan3_capped_{target_resolution}.pkl"
print(f"Saving capped StyleGAN3 generator to {stylegan3_output_pkl}...")

# Load the original network data to preserve other components
with dnnlib.util.open_url(stylegan3_pkl) as f:
    stylegan3_network_data = legacy.load_network_pkl(f)

# Save the generator
save_generator(G3_capped, stylegan3_output_pkl, stylegan3_network_data)
print(f"Capped StyleGAN3 generator saved to {stylegan3_output_pkl}")

In [ ]:
# Save the capped StyleGAN2 generator
stylegan2_output_pkl = f"stylegan2_capped_{target_resolution}.pkl"
print(f"Saving capped StyleGAN2 generator to {stylegan2_output_pkl}...")

# Load the original network data to preserve other components
with dnnlib.util.open_url(stylegan2_pkl) as f:
    stylegan2_network_data = legacy.load_network_pkl(f)

# Save the generator
save_generator(G2_capped, stylegan2_output_pkl, stylegan2_network_data)
print(f"Capped StyleGAN2 generator saved to {stylegan2_output_pkl}")

## Verify the Saved Models

In [ ]:
# Load the saved capped StyleGAN3 generator
print(f"Loading capped StyleGAN3 generator from {stylegan3_output_pkl}...")
with open(stylegan3_output_pkl, 'rb') as f:
    loaded_network_data = torch.load(f)
    G3_loaded = loaded_network_data['G_ema'].to(device)

print(f"Loaded StyleGAN3 generator resolution: {G3_loaded.img_resolution}")

# Generate an image with the loaded generator
seed = 300
z = torch.from_numpy(np.random.RandomState(seed).randn(1, G3_loaded.z_dim)).to(device)

img3_loaded = generate_images(G3_loaded, z, truncation_psi=0.7)
img3_loaded_display = (img3_loaded.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)

plt.figure(figsize=(10, 10))
plt.imshow(img3_loaded_display[0].cpu().numpy())
plt.title(f"Image from loaded capped StyleGAN3 ({G3_loaded.img_resolution}x{G3_loaded.img_resolution})")
plt.axis('off')
plt.show()

In [ ]:
# Load the saved capped StyleGAN2 generator
print(f"Loading capped StyleGAN2 generator from {stylegan2_output_pkl}...")
with open(stylegan2_output_pkl, 'rb') as f:
    loaded_network_data = torch.load(f)
    G2_loaded = loaded_network_data['G_ema'].to(device)

print(f"Loaded StyleGAN2 generator resolution: {G2_loaded.img_resolution}")

# Generate an image with the loaded generator
seed = 300
z = torch.from_numpy(np.random.RandomState(seed).randn(1, G2_loaded.z_dim)).to(device)

img2_loaded = generate_images(G2_loaded, z, truncation_psi=0.7)
img2_loaded_display = (img2_loaded.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)

plt.figure(figsize=(10, 10))
plt.imshow(img2_loaded_display[0].cpu().numpy())
plt.title(f"Image from loaded capped StyleGAN2 ({G2_loaded.img_resolution}x{G2_loaded.img_resolution})")
plt.axis('off')
plt.show()